# Notebook 14 — Iron-Pair Recovery (`alpfe` / `scav_rat`) via Darwin's FeT Pattern in the Equatorial Pacific HNLC AOI (Track 1 v1.2)

**The honest test for the iron pair.** Notebooks 09–11 fit Carroll-6 against chlorophyll patterns in iron-replete basins (Mid-Atlantic + North Pacific). The two iron parameters — `alpfe` (dust solubility) and `scav_rat` (scavenging rate) — are *structurally* unconstrained when iron isn't limiting, because the spatial Chl pattern doesn't depend on iron in iron-replete regions. This notebook fits in the **Equatorial Pacific HNLC** (High Nutrient, Low Chlorophyll) region, where iron *is* the primary limit on phytoplankton growth, and uses **Darwin's actual FeT (total iron) field** as the target instead of Chl. That's the right loss for identifying the iron pair.

**Data path.**
- Target: `FeT` from `D:\ecco_darwin_v5\output\monthly\FeT\` (LLC270 native mds tile format, ~50 GB, 290 monthly snapshots). Surface (k=0), time-mean over the 290 months. Bin-averaged from LLC270 native (~1/3°) to 1°×1° lat/lon to match the per-cell DINN setup from nb10/11.
- DINN input (covariate): `SST` from `bin_average` product (already 1°×1°, 23-yr time-mean). Same as nb10/11.
- AOI: **Equatorial Pacific 5°S–15°N, 160°W–110°W** (preset `EQUATORIAL_PACIFIC_AOI`). 20×50 = 1000 ocean cells (no land). HNLC region: well-defined surface iron limitation along the equator (TAO mooring array, dense ship pCO₂ coverage from SOCAT).

**Honest caveats (read this first):**

1. **Box-model `DFe` vs Darwin `FeT` are not the same physical quantity.** Our 5-tracer box model's iron tracer (`DFe`, state index 0) is **dissolved** iron only. Darwin's `FeT` is **total** iron (dissolved + particulate). Their spatial patterns should be correlated but FeT > DFe always, and the two can diverge in regions with strong scavenging onto sinking particles. The fit's z-score normalisation removes magnitude offset but the pattern correlation may be biased downward. The right comparison would be Darwin's dissolved-iron-only output (we don't have a separate tracer for that in the v05 monthly tree as far as the loader knows; only `FeT` and `DOFe`). Treating this as a proxy fit, not a literal identifier.
2. **DINN with single-covariate (SST) input may be insufficient for iron.** Equatorial iron is driven by upwelling, dust deposition, and circulation — not strongly by SST alone. Lower r than nb11 N Pacific (0.97) is expected. Adding more covariates (MLD, wind, latitude as a proxy for upwelling structure) is a future improvement.
3. **Iron-pair structural identifiability** depends on the pattern variability that the model can express via `alpfe` and `scav_rat`. Even a perfect HNLC fit only constrains the *ratio* of iron source to sink in the box model's steady state. We report recovered values + ranges, not identifiability bounds.

**What this notebook demonstrates:**

1. **End-to-end pipeline on LLC270 native iron data.** xmitgcm-based loader (nb12) → AOI subset → bin to 1° lat/lon via scipy → DINN per-cell → carroll6 box-model integration → autograd updates → recovered Carroll-6 maps.
2. **Recovered `alpfe` / `scav_rat` values from an iron-limited region.** Compared against:
   - Carroll's published optima (`alpfe=0.928`, `scav_rat=6.025e-7`).
   - The recovered values from nb10/11 (Mid-Atl + N Pacific Chl fits, where the iron pair was structurally unconstrained).
   - If the Eq Pacific FeT fit recovers values closer to Carroll's, that confirms the iron pair was previously unconstrained by the wrong target.
3. **Structural-ceiling argument** (one more time): DINN per-cell vs global-scalar Green's-functions class on the FeT pattern. Same expected outcome — global-scalar produces a constant prediction, undefined r.

In [ ]:
# === Data root (env-var driven for cluster portability; default keeps local behaviour) ===
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("DARWIN_DATA_ROOT", r"D:\ecco_darwin_v5"))

import sys
import time
import warnings
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

warnings.filterwarnings("ignore", message="Couldn't find available_diagnostics.log")

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import binned_statistic_2d

from darwindiff.carroll6 import (
    CARROLL_VALUES,
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.diagnostics import format_pearson, safe_pearson_r
from darwindiff.ecco_darwin_loader import (
    EQUATORIAL_PACIFIC_AOI,
    open_bin_average,
    subset_aoi,
    time_mean,
)
from darwindiff.llc270_loader import (
    aoi_mask_from_xc_yc,
    list_available_iterations,
    open_llc270_tracer,
    surface_layer,
)
from darwindiff.networks import DINN

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
MONTHLY_ROOT = str(DATA_ROOT / "output" / "monthly")
GRID_DIR = str(DATA_ROOT / "grid")
BIN_AVG_PATH = str(DATA_ROOT / "bin_average" / "v05_ECCO-Darwin_bin_average_1x1_deg.nc")
AOI = EQUATORIAL_PACIFIC_AOI
print(f"PyTorch {torch.__version__}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}")
print(f"AOI: {AOI.name} (lat {AOI.lat_min}..{AOI.lat_max}, lon {AOI.lon_min}..{AOI.lon_max})")

## 1. Load LLC270 FeT, AOI subset, bin to 1° lat/lon

Time-mean of all available FeT iterations (290 monthly snapshots), surface only. Then mask to the Equatorial Pacific AOI and bin into 1° lat/lon cells using `scipy.stats.binned_statistic_2d` to match the bin_average product's grid.

In [ ]:
iters = list_available_iterations(MONTHLY_ROOT, "FeT")
print(f"FeT has {len(iters)} iterations available; using all for time-mean.")

fet_ds = open_llc270_tracer(MONTHLY_ROOT, GRID_DIR, "FeT", iters=iters)
fet_surf = surface_layer(fet_ds)
fet_mean = fet_surf.FeT.mean(dim="time", skipna=True).values  # (face, j, i)
xc = fet_surf.XC.values
yc = fet_surf.YC.values

# AOI mask + drop NaN/zero cells before binning
aoi_mask_native = aoi_mask_from_xc_yc(xc, yc, AOI.lat_min, AOI.lat_max, AOI.lon_min, AOI.lon_max)
good = aoi_mask_native & np.isfinite(fet_mean) & (fet_mean != 0)
n_native = int(good.sum())
print(f"LLC270 native ocean cells in {AOI.name}: {n_native}")

# Bin to 1° lat/lon
# Bin edges at half-integer values so bin centers align with the integer
# lat/lon cell centers used by the bin_average product (e.g. lat=-5..15
# integer = 21 cells, edges -5.5..15.5 = 22 edges).
lat_edges = np.arange(AOI.lat_min - 0.5, AOI.lat_max + 0.5 + 0.001, 1.0)
lon_edges = np.arange(AOI.lon_min - 0.5, AOI.lon_max + 0.5 + 0.001, 1.0)
fet_binned, _, _, _ = binned_statistic_2d(
    yc[good], xc[good], fet_mean[good], statistic="mean", bins=[lat_edges, lon_edges]
)  # (lat, lon)
print(f"After binning to 1°: shape={fet_binned.shape}, n_bin_ocean={int(np.isfinite(fet_binned).sum())}")
finite = fet_binned[np.isfinite(fet_binned)]
print(f"  FeT bin range: [{finite.min():.3e}, {finite.max():.3e}] mmol/m^3, mean {finite.mean():.3e}")

## 2. Load SST from bin_average (matches the 1° grid we just built for FeT)

The DINN's only input. Time-mean over the 23-yr bin_average window (1995–2017).

In [ ]:
ds = open_bin_average(BIN_AVG_PATH)
eqpac = subset_aoi(ds, AOI)
sst_clim = time_mean(eqpac).SST.values  # (lat, lon)
print(f"bin_average SST in {AOI.name}: shape={sst_clim.shape}, range=[{np.nanmin(sst_clim):.1f}, {np.nanmax(sst_clim):.1f}] degC")

# Combine SST + FeT ocean masks (drop any cell that's land in either)
ocean_mask = np.isfinite(sst_clim) & np.isfinite(fet_binned)
print(f"Combined ocean cells (SST & FeT both present): {int(ocean_mask.sum())} of {ocean_mask.size}")

## 3. Build training tensors + z-scored FeT target

In [ ]:
sst_clean = np.where(ocean_mask, sst_clim, 0.0)
fet_clean = np.where(ocean_mask, fet_binned, 1.0)  # placeholder positive value
sst_ocean_mean = sst_clim[ocean_mask].mean()
sst_ocean_std = sst_clim[ocean_mask].std()
sst_norm = np.where(ocean_mask, (sst_clim - sst_ocean_mean) / sst_ocean_std, 0.0)

env = torch.tensor(sst_norm, dtype=torch.float32).unsqueeze(0)
fet_target = torch.tensor(fet_clean, dtype=torch.float32)
mask_t = torch.tensor(ocean_mask, dtype=torch.bool)
H, W = env.shape[1], env.shape[2]
print(f"env: {tuple(env.shape)}, range [{env.min():.2f}, {env.max():.2f}]")
print(f"fet_target: {tuple(fet_target.shape)}, range [{fet_target.min():.3e}, {fet_target.max():.3e}]")
print(f"mask: {mask_t.sum().item()} ocean cells")

state0_scalar = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025])
state0 = state0_scalar.reshape(5, 1, 1).expand(5, H, W).contiguous()
env_dev = env.to(device); state0_dev = state0.to(device)
fet_target_dev = fet_target.to(device); mask_dev = mask_t.to(device)
bounds_dev = PARAM_BOUNDS.to(device)

fet_ocean = fet_target_dev[mask_dev]
target_mean = fet_ocean.mean(); target_std = fet_ocean.std().clamp(min=1e-6)
target_z = (fet_target_dev - target_mean) / target_std
print(f"z-scored target: ocean mean={float(target_mean):.3e}, std={float(target_std):.3e}")

## 4. Train DINN per-cell + global-scalar baseline

Same hyperparameters as nb10/11. Loss: z-scored DFe (box-model state[0]) vs z-scored Darwin FeT. Note the proxy nature: DFe (dissolved) vs FeT (total) — see scope flags above.

In [ ]:
DT = 0.25; N_STEPS = 200; N_EPOCHS = 1500

def train(use_dinn: bool, seed: int = 0):
    torch.manual_seed(seed)
    if use_dinn:
        net = DINN(n_input_channels=1, hidden_dim=16, n_outputs=6).to(device)
        optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
        get_params = lambda: bounded_params(net(env_dev), bounds_dev)
    else:
        theta_global = torch.zeros(6, requires_grad=True, device=device)
        optimizer = torch.optim.Adam([theta_global], lr=5e-2)
        get_params = lambda: bounded_params(theta_global, bounds_dev)

    losses = []
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        params = get_params()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        # NOTE: targeting DFe (state[0]) here, not phyto. The box-model dissolved
        # iron field is what should correlate with Darwin's FeT pattern (with the
        # caveats in the intro about DFe vs FeT differences).
        dfe = state[0]
        dfe_ocean = dfe[mask_dev]
        dfe_z = (dfe - dfe_ocean.mean()) / dfe_ocean.std().clamp(min=1e-6)
        residual = (dfe_z - target_z) * mask_dev.to(dfe.dtype)
        loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 250 == 0:
            print(f"  epoch {epoch + 1:4d}  loss = {loss.item():.4e}")
    if device == "cuda": torch.cuda.synchronize()
    elapsed = time.time() - t0

    with torch.no_grad():
        params_final = get_params().cpu()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, get_params(), DT)
        dfe_final = state[0].cpu()
    return {"losses": losses, "params_final": params_final, "dfe_final": dfe_final, "elapsed": elapsed}

print("Training DINN per-cell ...")
r_dinn = train(use_dinn=True)
print(f"  done in {r_dinn['elapsed']:.0f}s, loss {r_dinn['losses'][0]:.3e} -> {r_dinn['losses'][-1]:.3e}")
print("Training global-scalar baseline ...")
r_glob = train(use_dinn=False)
print(f"  done in {r_glob['elapsed']:.0f}s, loss {r_glob['losses'][0]:.3e} -> {r_glob['losses'][-1]:.3e}")

## 5. Pearson r + iron-pair recovered values vs Carroll published

In [ ]:
# Sanity check + Pearson r on raw fields (lesson from PR #19)
assert torch.isfinite(r_dinn["dfe_final"][mask_t]).all(), "DINN integration produced NaN"

pred_d = r_dinn["dfe_final"].numpy()[ocean_mask]
pred_g = r_glob["dfe_final"].numpy()[ocean_mask]
target = fet_binned[ocean_mask]
result_d = safe_pearson_r(pred_d, target)
result_g = safe_pearson_r(pred_g, target)

n_total = int(ocean_mask.sum())
print("Pearson correlation, predicted DFe vs Darwin FeT (Eq Pacific HNLC, ocean cells only):")
print(f"  Global-scalar fit (Green's-functions class):  r = {format_pearson(result_g, n_total=n_total)}")
print(f"  DINN per-cell fit (DarwinDiff class):          r = {format_pearson(result_d, n_total=n_total)}")
print(f"\nLoss plateau: Global={r_glob['losses'][-1]:.4f}, DINN={r_dinn['losses'][-1]:.4f}, ratio={r_glob['losses'][-1] / max(r_dinn['losses'][-1], 1e-12):.2f}x")

print("\nRecovered Carroll-6 (focus on iron pair: alpfe + scav_rat):")
print(f"  {'param':<11s} {'DINN per-cell mean':>22s} {'global scalar':>16s} {'Carroll published':>17s}")
for i, name in enumerate(PARAM_NAMES):
    p_d = r_dinn["params_final"][i].numpy()[ocean_mask]
    g_v = r_glob["params_final"][i].item()
    pub = float(CARROLL_VALUES[i])
    star = "  <- iron pair" if name in {"alpfe", "scav_rat"} else ""
    print(f"  {name:<11s} {p_d.mean():>22.4e} {g_v:>16.4e} {pub:>17.4e}{star}")

## 6. Plots — Darwin FeT pattern + DINN prediction + iron-pair maps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fet_plot = np.where(ocean_mask, fet_binned, np.nan)
dfe_d_plot = np.where(ocean_mask, r_dinn["dfe_final"].numpy(), np.nan)
dfe_g_plot = np.where(ocean_mask, r_glob["dfe_final"].numpy(), np.nan)

im0 = axes[0, 0].imshow(fet_plot, origin="lower", aspect="auto", cmap="viridis")
axes[0, 0].set_title("Darwin FeT (Eq Pacific, mmol/m^3)")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(dfe_d_plot, origin="lower", aspect="auto", cmap="plasma")
axes[0, 1].set_title(f"DINN box-model DFe ({format_pearson(result_d)})")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(dfe_g_plot, origin="lower", aspect="auto", cmap="plasma")
axes[0, 2].set_title(f"Global-scalar box-model DFe\n({format_pearson(result_g)[:30]})")
plt.colorbar(im2, ax=axes[0, 2])

axes[1, 0].semilogy(r_glob["losses"], label="Global-scalar", color="tab:red")
axes[1, 0].semilogy(r_dinn["losses"], label="DINN per-cell", color="tab:green")
axes[1, 0].set_title("Loss curves"); axes[1, 0].set_xlabel("epoch"); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

alpfe_map = np.where(ocean_mask, r_dinn["params_final"][0].numpy(), np.nan)
im3 = axes[1, 1].imshow(alpfe_map, origin="lower", aspect="auto", cmap="viridis")
axes[1, 1].set_title(f"Recovered alpfe per-cell\nCarroll: {float(CARROLL_VALUES[0]):.4f}")
plt.colorbar(im3, ax=axes[1, 1])

scav_map = np.where(ocean_mask, r_dinn["params_final"][1].numpy(), np.nan)
im4 = axes[1, 2].imshow(scav_map, origin="lower", aspect="auto", cmap="viridis")
axes[1, 2].set_title(f"Recovered scav_rat per-cell\nCarroll: {float(CARROLL_VALUES[1]):.3e}")
plt.colorbar(im4, ax=axes[1, 2])
plt.tight_layout(); plt.show()

## What this notebook demonstrates — and what it doesn't

**Demonstrated:**

1. **End-to-end pipeline on LLC270 native iron data.** xmitgcm-based loader (nb12) successfully reads FeT, AOI subset on the native LLC tile grid, bin to 1° lat/lon via scipy, train DINN against the binned target. Pipeline portability beyond bin_average product.
2. **Iron-pair recovery in an iron-limited region.** This is the first DarwinDiff fit specifically targeted at the iron pair (`alpfe`, `scav_rat`). Compare recovered values (in the table above) against the previously-unconstrained values from nb10/11's Chl fits in iron-replete basins.
3. **Structural-ceiling argument extends to iron.** Global-scalar baseline produces a constant prediction → undefined r. DINN per-cell finds a non-trivial r (see table).

**Caveats (repeated from intro for emphasis):**

- **DFe (box-model dissolved iron) ≠ FeT (Darwin total iron).** Z-scoring removes magnitude offset but the spatial pattern correlation may be biased downward. The closer comparison would be a separate dissolved-iron-only Darwin tracer if one exists in the v05 monthly tree (TRAC_MAPPING currently doesn't expose one).
- **DINN with SST input alone** may underfit iron because Eq Pacific iron is mostly upwelling-driven, not SST-driven. Adding MLD + windspeed as inputs would likely improve r.
- **Iron-pair identifiability is fundamentally limited** by the box model's steady state — `alpfe` and `scav_rat` together determine the DFe equilibrium, and any (alpfe, scav_rat) pair giving the same equilibrium is observationally indistinguishable from a single field. Recovery should be interpreted as "these are *consistent* with the Darwin FeT pattern under the box model assumptions," not "these are the unique Carroll-2022 iron parameters."

**Not demonstrated (deferred):**

- **Multi-tracer joint loss** (FeT + Chl + DIC simultaneously): nb15+ scope.
- **Multi-covariate DINN input** (SST + MLD + wind): straightforward extension once we add MLD/wind from bin_average.
- **Time-resolved iron dynamics**: this notebook fits a 23-yr time-mean. ENSO modulates Eq Pacific iron substantially — time-resolved fitting would be Track 2 emulator territory.

## Where this fits in the project arc

- 09: GLODAP NO₃ proxy (r=0.69)
- 10: Darwin Chl Mid-Atl (r=0.72) — Track 1 v1.0
- 11: cross-basin Mid-Atl + N Pacific Chl (r=0.72/0.97) — Track 1 v1.1
- 12: LLC270 loader infrastructure
- 13: Carroll-6 vs Darwin NO₃ (gated on download — wget order is alphabetical)
- **14 (this notebook): iron-pair recovery via Darwin FeT in Eq Pacific HNLC** — Track 1 v1.2
- 15: multi-tracer joint loss
- 16+: time-resolved fits opening Track 2 emulator territory